# MP03 — Press Release to Plot: Industry Comparison
**CIS 3120 — Programming for Analytics · Baruch College, Zicklin School of Business**

| Role | Member |
|---|---|
| Financial Services Pipeline Lead | *Adrian D.* |
| Travel & Hospitality Pipeline Lead | *Amalie M.* |
| Comparison & Visualization Lead (Integrator) | *Sarah H.* |

**Team Number:** `12` — update before submission.

---
## 0. Setup & Dependencies
**Two manual steps required before running:**
1. Add your Anthropic API key to Colab Secrets under the name `ANTHROPIC_API_KEY`.
2. Replace the `USER_AGENT` placeholder string below with your actual name/email.

In [ ]:
# Install required packages
!pip install -q anthropic requests folium geopy pandas

In [ ]:
import os
import json
import time
import requests
import pandas as pd
import folium
from datetime import date, timedelta
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut
import anthropic

# ── API Key (Colab Secrets) ──────────────────────────────────────────────────
try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    # Fallback: set env var manually if not running in Colab
    ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")

assert ANTHROPIC_API_KEY, "❌ ANTHROPIC_API_KEY not found — add it to Colab Secrets."

# ── User-Agent (EDGAR requires a descriptive User-Agent) ─────────────────────
# Replace the placeholder with your real name and email before running.
USER_AGENT = "Adrian D. Adrian.Davis@baruch.cuny.edu CIS3120 MP03"

# ── Anthropic client ─────────────────────────────────────────────────────────
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

print("✅ Setup complete.")

---
## 1. Industry Ticker Lists & Search Phrases

The seeds below are the instructor-provided defaults. Both lists have been extended with justified additions documented in the Methodology section (Section 6).

In [ ]:
# ── Financial Services Tickers ───────────────────────────────────────────────
# Seed: 14 companies across money-center banks, regional banks, asset mgmt,
# insurance, and payments.
# Extensions: Added capital-markets (GS, MS), fintech (SQ, PYPL), and an
# additional regional bank (RF) to broaden geographic coverage and capture
# capital-markets location events underrepresented in the retail-banking seed.
FINANCIAL_SERVICES_TICKERS = [
    # Money-center banks
    "JPM", "BAC", "WFC", "C",
    # Capital markets (extension — seed was biased toward retail banking)
    "GS", "MS",
    # Regional banks
    "PNC", "USB", "TFC", "RF",
    # Asset management
    "BLK", "BX",
    # Insurance
    "MET", "PRU",
    # Payments / Fintech (extension — growing footprint of physical ops centers)
    "V", "MA", "AXP", "SQ", "PYPL",
]

# ── Financial Services Search Phrases ────────────────────────────────────────
# Extensions: Added capital-markets-specific phrases to capture trading floor
# and advisory office moves, and "technology center" for the wave of bank tech
# hub openings.
FINANCIAL_SERVICES_PHRASES = [
    '"new branch"',
    '"branch opening"',
    '"branch closure"',
    '"branch closing"',
    '"branch consolidation"',
    '"regional office"',
    '"office closure"',
    '"operations center"',
    '"data center"',
    '"new location"',
    # Extensions
    '"technology center"',
    '"trading floor"',
    '"advisory office"',
    '"wealth management office"',
]

# ── Travel & Hospitality Tickers ─────────────────────────────────────────────
# Seed: 14 companies across hotels, cruise, airlines, and online travel.
# Extensions: Added resort/casino operators (MGM, WYNN, LVS) and a budget
# airline (JBLU) to capture leisure-demand expansion events beyond the
# major-carrier / major-chain seed.
TRAVEL_HOSPITALITY_TICKERS = [
    # Hotels
    "MAR", "HLT", "H", "CHH", "WH",
    # Cruise
    "CCL", "RCL", "NCLH",
    # Airlines
    "DAL", "UAL", "AAL", "LUV",
    # Budget / regional airlines (extension)
    "JBLU",
    # Online travel
    "BKNG", "EXPE",
    # Resort / Casino operators (extension — major physical-expansion filers)
    "MGM", "WYNN", "LVS",
]

# ── Travel & Hospitality Search Phrases ──────────────────────────────────────
# Extensions: Added cruise-port and casino-specific terms, and separated
# hotel-style from airline-style phrases to improve Stage 3 precision.
TRAVEL_HOSPITALITY_PHRASES = [
    '"new property"',
    '"new hotel"',
    '"hotel opening"',
    '"resort opening"',
    '"property opening"',
    '"brand conversion"',
    '"new route"',
    '"new gateway"',
    '"new terminal"',
    '"grand opening"',
    # Extensions
    '"new destination"',
    '"casino opening"',
    '"homeport"',
    '"new port"',
    '"resort expansion"',
]

print(f"Financial Services: {len(FINANCIAL_SERVICES_TICKERS)} tickers, {len(FINANCIAL_SERVICES_PHRASES)} phrases")
print(f"Travel & Hospitality: {len(TRAVEL_HOSPITALITY_TICKERS)} tickers, {len(TRAVEL_HOSPITALITY_PHRASES)} phrases")

---
## 2. Preserved Module 15 Function Signatures
These five signatures are unchanged from the instructor notebook. Do not modify.

In [ ]:
# ── Stage 1: EDGAR full-text search ─────────────────────────────────────────
def search_edgar_one_phrase(
    phrase: str,
    start_date: date,
    end_date: date,
    forms: str = "8-K",
    max_pages: int = 2,
) -> tuple[list[dict], int]:
    """
    Search EDGAR full-text search for 8-K filings containing `phrase`
    within the given date range. Returns (hits, total_count).
    """
    base_url = "https://efts.sec.gov/LATEST/search-index?q={q}&dateRange=custom&startdt={s}&enddt={e}&forms={f}"
    hits = []
    total = 0
    headers = {"User-Agent": USER_AGENT}

    for page in range(max_pages):
        url = (
            f"https://efts.sec.gov/LATEST/search-index?q={requests.utils.quote(phrase)}"
            f"&dateRange=custom&startdt={start_date}&enddt={end_date}"
            f"&forms={forms}&from={page * 10}&size=10"
        )
        try:
            resp = requests.get(url, headers=headers, timeout=15)
            resp.raise_for_status()
            data = resp.json()
            hits_page = data.get("hits", {}).get("hits", [])
            if page == 0:
                total = data.get("hits", {}).get("total", {}).get("value", 0)
            hits.extend(hits_page)
            if not hits_page:
                break
            time.sleep(0.5)  # be polite to EDGAR
        except Exception as e:
            print(f"  [EDGAR] Error on phrase '{phrase}', page {page}: {e}")
            break

    return hits, total


# ── Stage 2: Build exhibit URL ───────────────────────────────────────────────
def build_exhibit_url(hit: dict) -> str:
    """
    Construct the SEC EDGAR viewer URL for the primary document in a hit.
    """
    source = hit.get("_source", {})
    accession = source.get("file_date", "")  # used as fallback label only
    entity_id = source.get("entity_id", "")
    file_num = source.get("file_num", "")
    accession_no = source.get("accession_no", "").replace("-", "")
    doc_name = source.get("file_name", "")

    if accession_no and doc_name:
        return f"https://www.sec.gov/Archives/edgar/data/{entity_id}/{accession_no}/{doc_name}"
    elif accession_no:
        clean = source.get("accession_no", "")
        return f"https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&filenum={file_num}&type=8-K"
    return ""


# ── Stage 2: Fetch exhibit text ──────────────────────────────────────────────
def fetch_exhibit_text(hit: dict, max_chars: int = 8000) -> tuple[str, str]:
    """
    Fetch the raw text of the primary document for a hit.
    Returns (text, url). Truncates to max_chars to stay within token budget.
    """
    url = build_exhibit_url(hit)
    if not url:
        return "", ""
    try:
        headers = {"User-Agent": USER_AGENT}
        resp = requests.get(url, headers=headers, timeout=20)
        resp.raise_for_status()
        text = resp.text[:max_chars]
        return text, url
    except Exception as e:
        print(f"  [fetch] Could not retrieve {url}: {e}")
        return "", url


# ── Stage 3: Claude extraction ───────────────────────────────────────────────
# Cost tracking accumulator (global, reset between tuning trials)
_trial_input_tokens = 0
_trial_output_tokens = 0

HAIKU_INPUT_COST_PER_TOKEN  = 1.00 / 1_000_000   # $1 per million input tokens
HAIKU_OUTPUT_COST_PER_TOKEN = 5.00 / 1_000_000   # $5 per million output tokens

def extract_with_claude(filing: dict) -> dict:
    """
    Use Claude Haiku to extract location event data from a filing's exhibit text.
    Returns a dict with keys: is_location_event, event_type, city, state, summary.
    Also accumulates token usage into module-level counters for cost tracking.
    """
    global _trial_input_tokens, _trial_output_tokens

    text  = filing.get("text", "")
    ticker = filing.get("ticker", "unknown")

    prompt = f"""You are a financial document analyst. Read the following SEC 8-K press release excerpt and extract location event information.

Company ticker: {ticker}

Document text:
{text}

Respond in JSON only (no markdown fences) with exactly these fields:
{{
  "is_location_event": true or false,
  "event_type": "opening" | "closure" | "relocation" | "expansion" | "route_launch" | "other" | null,
  "city": "City name or null",
  "state": "Two-letter state code or null",
  "summary": "One sentence description or null"
}}

Set is_location_event to true only if the filing announces a specific physical location change (opening, closure, relocation, expansion, or new route). If it is a generic press release with no specific location event, set is_location_event to false and all other fields to null."""

    try:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=256,
            messages=[{"role": "user", "content": prompt}],
        )
        # Accumulate token usage for cost estimation
        _trial_input_tokens  += response.usage.input_tokens
        _trial_output_tokens += response.usage.output_tokens

        raw = response.content[0].text.strip()
        # Strip accidental markdown fences
        raw = raw.replace("```json", "").replace("```", "").strip()
        return json.loads(raw)
    except json.JSONDecodeError as e:
        print(f"  [Claude] JSON parse error for {ticker}: {e}")
        return {"is_location_event": False, "event_type": None,
                "city": None, "state": None, "summary": None}
    except Exception as e:
        print(f"  [Claude] API error for {ticker}: {e}")
        return {"is_location_event": False, "event_type": None,
                "city": None, "state": None, "summary": None}


# ── Stage 4: Geocoding ────────────────────────────────────────────────────────
_geocoder = Nominatim(user_agent=USER_AGENT)
_geocode_cache: dict[tuple, tuple | None] = {}

def geocode_location(city: str, state: str | None) -> tuple[float, float] | None:
    """
    Geocode a city (and optionally state) to (latitude, longitude).
    Results are cached to avoid redundant API calls.
    Returns None if geocoding fails.
    """
    key = (city, state)
    if key in _geocode_cache:
        return _geocode_cache[key]

    query = f"{city}, {state}, USA" if state else f"{city}, USA"
    try:
        loc = _geocoder.geocode(query, timeout=10)
        if loc:
            result = (loc.latitude, loc.longitude)
        else:
            result = None
        _geocode_cache[key] = result
        time.sleep(1)  # Nominatim rate limit
        return result
    except GeocoderTimedOut:
        print(f"  [Geocode] Timed out for '{query}'")
        _geocode_cache[key] = None
        return None


print("✅ Preserved Module 15 functions loaded.")